# Tugas Terstruktur 3

## Metrik Evaluasi dan Pemilihan Ambang Batas

**Mata Kuliah** Data Science (TI24425) &middot; 3 sks (Teori)
**Program Studi** Teknologi Informasi &middot; Politeknik Negeri Madiun
**Semester** Genap &middot; Tahun Akademik 2026/2027
**Cakupan materi** Pertemuan 6 dan 7
**Bobot** 4% dari nilai akhir
**Bentuk** Kerja kelompok, tiga orang &middot; diberikan pada pertemuan 7

**Materi rujukan** Pertemuan 6 dan 7 &middot; **Sub-CPMK 7**

---

### Identitas Kelompok

| | Nama Lengkap | NPM | Peran | Bagian |
|---|---|---|---|---|
| 1 | | | Penghitung Metrik | Bagian A |
| 2 | | | Pengevaluasi Model | Bagian B |
| 3 | | | Penimbang Biaya Kesalahan | Bagian C |

| | |
|---|---|
| **Kelas** | |
| **Judul dataset** | |
| **Tanggal pengumpulan** | |


---

## Petunjuk Pengerjaan

### Kesinambungan dengan tugas sebelumnya

Tugas ini memakai **dataset yang sama** dengan Tugas Terstruktur 1. Jangan berganti dataset. Seluruh rangkaian tugas terstruktur dirancang menumpuk, dan hasilnya menjadi bekal langsung bagi Studi Kasus Akhir pada pertemuan 14 dan 15.

### Cara mengerjakan

- Setiap anggota mengerjakan **satu bagian** sesuai perannya. **Bagian D dikerjakan bersama.**
- Sel bertanda `[KODE]` diisi kode Python. Sel bertanda `[URAIAN]` diisi tulisan Anda sendiri.
- **Kode yang berjalan tanpa uraian tidak memperoleh nilai.** Yang dinilai adalah penalaran.
- Jangan menghapus sel pertanyaan. Tulis jawaban tepat di bawahnya.

### Status kode pada mata kuliah teori

Mata kuliah ini adalah mata kuliah teori. Kegiatan berbasis komputer berlangsung di luar jam tatap muka sebagai **Penugasan Terstruktur**. Kode yang diminta sengaja dibuat sederhana dan dapat dikerjakan dengan menyesuaikan nama kolom. Yang dinilai tetap kualitas analisis.

### Menyimpan sebagai PDF

1. Jalankan seluruh sel: **Run &rarr; Run All Cells**. Pastikan tidak ada pesan galat.
2. **File &rarr; Print** (`Ctrl` + `P`), pilih tujuan **Save as PDF**.
3. Beri nama `TT3_<Kelas>_<NamaKelompok>.pdf`, unggah bersama berkas `.ipynb` ke LMS.

### Penggunaan AI generatif

Diperbolehkan sebagai alat bantu, dengan syarat dicantumkan pada Lampiran di akhir berkas: bagian mana yang dibantu dan bagaimana hasilnya Anda verifikasi.


> **Catatan.** Bila dataset kelompok Anda berupa masalah **regresi**, buatlah kolom target biner buatan untuk keperluan tugas ini — misalnya “nilai di atas median” — dan jelaskan pada Bagian B bagaimana Anda membentuknya serta apa konsekuensinya.

---

## Persiapan

In [ ]:
# [KODE] Persiapan pustaka — jalankan apa adanya
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

pd.set_option('display.max_columns', 50); pd.set_option('display.width', 120)
plt.rcParams['figure.figsize'] = (7.5, 4); plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.grid'] = True; plt.rcParams['grid.alpha'] = 0.3
np.random.seed(42)
print('Pustaka siap.')

In [ ]:
# [KODE] Pemuatan dan penyiapan data — sesuaikan tiga baris pertama saja
NAMA_BERKAS  = 'nama_berkas_dataset.csv'
PEMISAH      = ','
KOLOM_TARGET = 'ganti_dengan_nama_kolom_target'

df = pd.read_csv(NAMA_BERKAS, sep=PEMISAH)

def siapkan(df, target, maks_kategori=15):
    """Menyiapkan X dan y: buang baris tanpa target, encoding kategorik, isi nilai hilang numerik."""
    d = df.dropna(subset=[target]).copy()
    y = d[target]
    X = d.drop(columns=[target])
    # buang kolom kategorik dengan terlalu banyak nilai unik (kemungkinan penanda identitas)
    buang = [c for c in X.columns
             if X[c].dtype == 'object' and X[c].nunique() > maks_kategori]
    if buang:
        print('Kolom dikeluarkan karena terlalu banyak nilai unik:', buang)
        X = X.drop(columns=buang)
    X = pd.get_dummies(X, drop_first=True)              # one-hot untuk kategorik
    X = X.fillna(X.median(numeric_only=True))            # isi nilai hilang numerik
    return X, y

X, y = siapkan(df, KOLOM_TARGET)
print('Ukuran X :', X.shape)
print('Ukuran y :', y.shape)
print('Tipe target:', 'kategorik / klasifikasi' if y.nunique() <= 10 else 'numerik / regresi')

---
---

# Bagian A &mdash; Perhitungan Metrik dari Confusion Matrix

**Dikerjakan oleh Anggota 1 &middot; Penghitung Metrik**

Bagian ini memakai data yang disediakan dosen. **Kerjakan perhitungannya dengan tangan lebih dulu**, baru periksa dengan kode.

## A.1 Dua Model pada Masalah yang Sama

Dari **1.000** perjalanan, **60** di antaranya benar-benar terlambat. Dua model diuji.

| | Model A | Model B |
|---|---|---|
| True Positive (TP) | 27 | 51 |
| False Negative (FN) | 33 | 9 |
| False Positive (FP) | 18 | 210 |
| True Negative (TN) | 922 | 730 |

**[URAIAN A.1]** Hitung dengan tangan, tulis langkah perhitungannya, lalu isi tabel berikut. Bulatkan tiga angka di belakang koma.

| Metrik | Rumus | Model A | Model B |
|---|---|---|---|
| Accuracy | | | |
| Precision | | | |
| Recall | | | |
| F1-score | | | |

> _Tulis langkah perhitungan Anda di sini._


In [ ]:
# [KODE] Memeriksa perhitungan tangan Anda
def metrik(TP, FN, FP, TN):
    total = TP + FN + FP + TN
    acc = (TP + TN) / total
    prec = TP / (TP + FP) if (TP + FP) else float('nan')
    rec = TP / (TP + FN) if (TP + FN) else float('nan')
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else float('nan')
    return dict(accuracy=round(acc, 3), precision=round(prec, 3),
                recall=round(rec, 3), f1=round(f1, 3))

banding = pd.DataFrame({
    'Model A': metrik(TP=27, FN=33, FP=18, TN=922),
    'Model B': metrik(TP=51, FN=9,  FP=210, TN=730),
})
print(banding)

**[URAIAN A.2]** Jawab keempat pertanyaan berikut.

1. Model mana yang **akurasinya** lebih tinggi? Apakah itu berarti model tersebut lebih baik? Jelaskan.
2. Model mana yang lebih banyak **menemukan keterlambatan yang sebenarnya**? Metrik mana yang menunjukkannya?
3. Hitung akurasi sebuah **model malas** yang selalu menjawab “tidak terlambat”. Bandingkan dengan Model A. Apa kesimpulan Anda tentang akurasi pada data tidak seimbang?
4. Bila Anda Kepala Dinas Perhubungan yang bertanggung jawab atas kepercayaan penumpang, model mana yang Anda pilih, dan **apa harga** yang harus Anda bayar untuk pilihan itu?

> _Tulis jawaban Anda di sini._


---
---

# Bagian B &mdash; Evaluasi Model pada Dataset Kelompok

**Dikerjakan oleh Anggota 2 &middot; Pengevaluasi Model**

In [ ]:
# [KODE] Melatih model klasifikasi dan menyusun confusion matrix
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (confusion_matrix, classification_report,
                             roc_curve, roc_auc_score, precision_recall_curve)

# Bila target Anda numerik, ubah menjadi biner — jelaskan alasannya pada uraian
if y.nunique() > 10:
    ambang_biner = y.median()
    y_bin = (y > ambang_biner).astype(int)
    print(f'Target diubah menjadi biner pada ambang median = {ambang_biner}')
else:
    y_bin = pd.factorize(y)[0]

X_latih, X_uji, y_latih, y_uji = train_test_split(
    X, y_bin, test_size=0.25, random_state=42, stratify=y_bin)

pipa = Pipeline([('skala', StandardScaler()),
                 ('model', LogisticRegression(max_iter=2000))])
pipa.fit(X_latih, y_latih)

peluang = pipa.predict_proba(X_uji)[:, 1]
tebakan = (peluang >= 0.5).astype(int)

cm = confusion_matrix(y_uji, tebakan)
print('Confusion matrix pada ambang 0,50')
print(pd.DataFrame(cm,
      index=['Sebenarnya 0', 'Sebenarnya 1'],
      columns=['Diprediksi 0', 'Diprediksi 1']))
print()
print('Proporsi kelas positif pada data uji :', round(y_uji.mean(), 4))
print()
print(classification_report(y_uji, tebakan, digits=3))

In [ ]:
# [KODE] Kurva ROC dan kurva Precision-Recall
fpr, tpr, _ = roc_curve(y_uji, peluang)
pre, rec, _ = precision_recall_curve(y_uji, peluang)
auc = roc_auc_score(y_uji, peluang)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].plot(fpr, tpr, lw=2); axes[0].plot([0, 1], [0, 1], ls='--', lw=1, color='gray')
axes[0].set_xlabel('False Positive Rate'); axes[0].set_ylabel('True Positive Rate')
axes[0].set_title(f'Kurva ROC · AUC = {auc:.3f}')
axes[1].plot(rec, pre, lw=2)
axes[1].axhline(y_uji.mean(), ls='--', lw=1, color='gray')
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].set_title('Kurva Precision-Recall')
plt.tight_layout(); plt.show()

**[URAIAN B.1]** Jawab kelima pertanyaan berikut.

1. Berapa proporsi kelas positif pada data uji Anda? Apakah datanya seimbang atau timpang?
2. Baca confusion matrix Anda: berapa **peringatan palsu** dan berapa **kasus yang terlewat**? Mana yang lebih banyak?
3. Berapa nilai AUC-nya, dan apa artinya? Jelaskan bahwa AUC mengukur **pemeringkatan**, bukan keputusan pada satu ambang tertentu.
4. Pada kurva Precision-Recall, terdapat garis putus-putus mendatar. Garis itu menandakan kinerja tebakan acak. Apakah kurva model Anda berada jelas di atasnya?
5. Bila target Anda diubah menjadi biner dari kolom numerik, jelaskan bagaimana Anda membentuknya dan **informasi apa yang hilang** akibat pengubahan itu.

> _Tulis jawaban Anda di sini._


---
---

# Bagian C &mdash; Ambang Batas dan Biaya Kesalahan

**Dikerjakan oleh Anggota 3 &middot; Penimbang Biaya Kesalahan**

In [ ]:
# [KODE] Pengaruh pergeseran ambang batas
baris = []
for t in [0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80]:
    pred = (peluang >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_uji, pred, labels=[0, 1]).ravel()
    prec = tp / (tp + fp) if (tp + fp) else np.nan
    rec = tp / (tp + fn) if (tp + fn) else np.nan
    f1 = 2 * prec * rec / (prec + rec) if (prec and rec) else np.nan
    baris.append({'ambang': t, 'peringatan_palsu_FP': fp, 'terlewat_FN': fn,
                  'precision': round(prec, 3) if prec == prec else None,
                  'recall': round(rec, 3) if rec == rec else None,
                  'f1': round(f1, 3) if f1 == f1 else None})

tabel_ambang = pd.DataFrame(baris)
print(tabel_ambang.to_string(index=False))

**[URAIAN C.1]** Jawab keempat pertanyaan berikut berdasarkan tabel di atas.

1. Bagaimana perubahan **FP** dan **FN** saat ambang dinaikkan dari 0,20 ke 0,80? Jelaskan mengapa keduanya bergerak berlawanan.
2. Adakah ambang yang membuat **kedua** jenis kesalahan sekaligus mengecil? Bila tidak ada, apa yang hal itu tunjukkan tentang sifat ambang batas?
3. Pada ambang berapa nilai **F1** paling tinggi? Apakah ambang itu otomatis merupakan pilihan terbaik? Jelaskan.
4. Sebutkan **satu keadaan nyata** pada konteks dataset Anda ketika Anda akan sengaja memilih ambang yang F1-nya **bukan** yang tertinggi.

> _Tulis jawaban Anda di sini._


## C.2 Menimbang Biaya Kesalahan Secara Eksplisit

Pada pertemuan 7 dibahas bahwa pemilihan metrik adalah keputusan berdasarkan biaya kesalahan, bukan kebiasaan.

**[URAIAN C.2]** Isi tabel berikut untuk konteks dataset kelompok Anda.

| Butir | Isian |
|---|---|
| Apa yang terjadi bila model menghasilkan **peringatan palsu** (FP)? Siapa yang menanggung? | |
| Apa yang terjadi bila model **melewatkan** kasus nyata (FN)? Siapa yang menanggung? | |
| Mana yang lebih merugikan, dan mengapa? | |
| Berdasarkan itu, metrik utama yang kelompok kami pilih | |
| Metrik yang justru **berbahaya** bila dipakai di sini, beserta alasannya | |
| Ambang batas yang kelompok kami usulkan | |

**[URAIAN C.3]** Ketiga konteks berikut memakai model yang sama persis, tetapi penggunanya berbeda. Untuk masing-masing, tentukan metrik utama dan arah ambang batas yang tepat, disertai alasan.

| Konteks | Metrik utama | Ambang dinaikkan atau diturunkan? | Alasan |
|---|---|---|---|
| Penapisan awal penumpang bergejala penyakit menular di terminal | | | |
| Sistem otomatis yang memblokir akun penumpang yang diduga menyalahgunakan kartu langganan | | | |
| Model yang memprediksi kerusakan mesin bus, dengan hanya 1,2% data berlabel rusak | | | |

> _Tulis jawaban Anda di sini._


---
---

# Bagian D &mdash; Sintesis Kelompok

**Dikerjakan bersama oleh ketiga anggota.**

## D.1 Laporan Kinerja yang Jujur

**[URAIAN D.1]** Susun satu paragraf berisi laporan kinerja model kelompok Anda **sebagaimana akan Anda sampaikan kepada pengguna yang bukan orang teknis**. Wajib memuat: metrik utama beserta angkanya, apa arti angka itu dalam kejadian nyata, dan satu keterbatasan yang Anda akui.

> _Tulis jawaban Anda di sini._


## D.2 Empat Pernyataan yang Harus Dinilai

**[URAIAN D.2]** Untuk setiap pernyataan, tentukan benar atau keliru, dan jelaskan mengapa.

| Pernyataan | Benar / keliru | Alasan |
|---|---|---|
| “Akurasi model kami 97%, jadi model ini sudah layak dipakai.” | | |
| “Kami memakai F1 karena F1 adalah metrik yang paling aman.” | | |
| “AUC kami 0,95, jadi keputusan model ini dapat dipercaya.” | | |
| “Kami menaikkan ambang ke 0,9 sehingga peringatan palsu menjadi nol. Masalah selesai.” | | |


## D.3 Pembagian Kerja

| Anggota | Bagian | Perkiraan waktu | Kesulitan terbesar |
|---|---|---|---|
| 1 | Bagian A | | |
| 2 | Bagian B | | |
| 3 | Bagian C | | |
| Bersama | Bagian D | | |


---

# Lampiran &mdash; Pernyataan Penggunaan AI Generatif

| Bagian yang dibantu | Nama alat | Bentuk bantuan | Cara kami memverifikasi |
|---|---|---|---|
| | | | |
| | | | |

Dengan ini kami menyatakan bahwa seluruh analisis, justifikasi, dan kesimpulan dalam berkas ini merupakan hasil penalaran kelompok kami sendiri, dan seluruh bantuan alat AI generatif telah kami cantumkan secara jujur.

| Anggota 1 | Anggota 2 | Anggota 3 |
|---|---|---|
| ( ......................... ) | ( ......................... ) | ( ......................... ) |


---

# Rubrik Penilaian

Total 100 poin, dikonversi menjadi bobot 4% pada nilai akhir.

| Bagian | Kriteria | Poin |
|---|---|---|
| A | Ketepatan perhitungan tangan keempat metrik untuk kedua model | 15 |
| A | Ketepatan menjelaskan mengapa akurasi menyesatkan pada data timpang | 10 |
| B | Ketepatan membaca confusion matrix dan proporsi kelas | 12 |
| B | Ketepatan menafsirkan AUC dan kurva Precision-Recall | 10 |
| B | Kejujuran menjelaskan pengubahan target menjadi biner, bila dilakukan | 5 |
| C | Ketepatan menafsirkan pergeseran ambang terhadap FP dan FN | 13 |
| C | Kekuatan justifikasi pemilihan metrik berdasarkan biaya kesalahan | 15 |
| C | Ketepatan menentukan metrik untuk tiga konteks berbeda | 10 |
| D | Mutu laporan kinerja untuk pengguna non-teknis dan penilaian empat pernyataan | 10 |
| | **Jumlah** | **100** |

### Ketentuan penilaian

- **Kode yang berjalan tanpa uraian bernilai nol** untuk butir yang bersangkutan.
- Jawaban umum yang dapat dipakai untuk dataset mana pun **tidak memperoleh nilai penuh**.
- Menyebut suatu hal keliru **tanpa menjelaskan mengapa** hanya memperoleh separuh poin.
- Mengakui keterbatasan secara jujur **menambah** nilai; menutupinya mengurangi nilai.
- Keterlambatan dikenakan pengurangan 10% nilai per hari kerja, maksimal tiga hari kerja.
